# DUA Edu · Nạp dữ liệu vào SQL Server

Đọc bốn CSV của challenge và kiểm tra dữ liệu trước khi nạp. Notebook gọi các script trong thư mục `sql/` để tạo bảng và view. Các timestamp được đọc theo giờ Việt Nam.

Đặt CSV trong `data/raw/` và chạy từ thư mục gốc dự án. Chế độ mặc định chỉ kiểm tra CSV. Bật `RUN_IMPORT` để nạp dữ liệu hoặc `RUN_VIEW_REFRESH` để cập nhật view.

## 1. Cấu hình — kiểm tra trước khi chạy

In [ ]:
from pathlib import Path
from datetime import date
from contextlib import closing
import csv
import re
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "sql" / "01_schema.sql").exists():
    raise FileNotFoundError("Hãy mở notebook và chạy từ thư mục gốc DUA_Edu_Clean_Project")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SQL_SERVER = r".\SQLEXPRESS"  # Đổi theo tên instance SQL Server của bạn
DATABASE = "DUA_Edu_Challenge"
COVERED_THROUGH = date(2026, 11, 10)

# CHẾ ĐỘ AN TOÀN — khi chạy lần đầu, tất cả mặc định False.
RUN_IMPORT = False           # True: tạo/nạp đủ 4 bảng từ CSV
REPLACE_EXISTING = False     # True: cho phép thay dữ liệu trong database ĐÍCH
CONFIRM_REPLACEMENT = ""    # Khi REPLACE_EXISTING=True: gõ đúng DUA_Edu_Challenge
RUN_VIEW_REFRESH = False     # True: chỉ cập nhật 3 view trên database hiện tại
RUN_EXPORT = False           # True: truy vấn view để xuất user_metrics.csv

print("Thư mục project:", PROJECT_DIR)
print("Database đích:", DATABASE)
print("Chế độ:", "IMPORT" if RUN_IMPORT else "READ-ONLY DATA CHECK")

## 2. Đọc đầy đủ 4 CSV gốc và kiểm tra cột

Tên file chấp nhận: `DUA Edu_Intern - Users.csv` hoặc `Users.csv` (tương tự Events/Content/Campaigns). Dữ liệu gốc không bị thay đổi.

In [ ]:
SOURCE_NAMES = ("Users", "Events", "Content", "Campaigns")
EXPECTED_COLUMNS = {
    "Users": ("user_id", "registered_at", "acquisition_channel", "device_type", "learning_goal", "experience_level", "ho_ten", "tinh_thanh"),
    "Events": ("event_id", "user_id", "occurred_at", "event_name", "content_id", "campaign_id"),
    "Content": ("content_id", "content_title", "topic", "level"),
    "Campaigns": ("campaign_id", "campaign_name", "campaign_type", "start_date", "end_date"),
}

def find_csv(name):
    for filename in (f"DUA Edu_Intern - {name}.csv", f"{name}.csv", f"{name.lower()}.csv"):
        candidate = RAW_DIR / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Thiếu CSV nguồn {name} trong {RAW_DIR}")

raw = {}
for name in SOURCE_NAMES:
    frame = pd.read_csv(find_csv(name), encoding="utf-8-sig", dtype=str, keep_default_na=False)
    if name == "Users":
        aliases = {"name": "ho_ten", "province": "tinh_thanh"}
        frame = frame.rename(columns={col: field for col, field in aliases.items() if field not in frame.columns})
    missing = set(EXPECTED_COLUMNS[name]) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} thiếu cột: {sorted(missing)}")
    raw[name] = frame.loc[:, list(EXPECTED_COLUMNS[name])].copy()
    print(f"{name:<10} {len(raw[name]):>6,} bản ghi · {len(raw[name].columns)} cột")

## 3. Data Quality và chuẩn hóa có kiểm soát

Báo lỗi thay vì âm thầm xóa hàng. `content_id`/`campaign_id` để trống là hợp lệ với nhiều loại sự kiện. Bốn bảng nguồn đều có kiểm tra khóa chính; sự kiện kiểm tra FK và timestamp.

In [ ]:
EVENT_NAMES = {
    "sign_up", "log_in", "view_content", "start_lesson", "complete_lesson",
    "submit_quiz", "check_in", "join_campaign", "share_content"
}
LESSON_EVENTS = {"view_content", "start_lesson", "complete_lesson"}
PRIMARY_KEYS = {"Users": "user_id", "Events": "event_id", "Content": "content_id", "Campaigns": "campaign_id"}

def prepare_source(raw_tables, covered_through):
    tables = {name: frame.copy() for name, frame in raw_tables.items()}
    for name, frame in tables.items():
        key = PRIMARY_KEYS[name]
        if frame.empty:
            raise ValueError(f"{name}: nguồn trống")
        frame[key] = frame[key].str.strip()
        if frame[key].eq("").any() or frame[key].duplicated().any():
            raise ValueError(f"{name}: {key} trống hoặc trùng")

    users, events = tables["Users"], tables["Events"]
    content, campaigns = tables["Content"], tables["Campaigns"]
    for name, columns in {
        "Users": ["acquisition_channel", "device_type", "learning_goal", "experience_level"],
        "Events": ["user_id", "event_name"],
        "Content": ["content_title", "topic", "level"],
        "Campaigns": ["campaign_name", "campaign_type", "start_date", "end_date"],
    }.items():
        for field in columns:
            if tables[name][field].astype(str).str.strip().eq("").any():
                raise ValueError(f"{name}.{field}: giá trị bắt buộc bị thiếu")

    for table, column in ((users, "registered_at"), (events, "occurred_at")):
        table[column] = pd.to_datetime(table[column], errors="raise")
        if table[column].isna().any() or table[column].dt.tz is not None:
            raise ValueError(f"{column}: timestamp thiếu hoặc có múi giờ ngoài dự kiến")

    for column in ("start_date", "end_date"):
        campaigns[column] = pd.to_datetime(campaigns[column], errors="raise").dt.date
    if (campaigns["end_date"] < campaigns["start_date"]).any():
        raise ValueError("Campaign có end_date trước start_date")

    for column in ("user_id", "content_id", "campaign_id", "event_name"):
        events[column] = events[column].astype(str).str.strip()
    if not events["user_id"].isin(users["user_id"]).all():
        raise ValueError("Events tham chiếu user_id không tồn tại")
    if not events["event_name"].isin(EVENT_NAMES).all():
        raise ValueError("Events có event_name ngoài danh sách đã định nghĩa")
    for column, table_name in (("content_id", "Content"), ("campaign_id", "Campaigns")):
        ids = events.loc[events[column].ne(""), column]
        if not ids.isin(tables[table_name][column]).all():
            raise ValueError(f"Events có {column} không tồn tại")
    if events.loc[events["event_name"].isin(LESSON_EVENTS), "content_id"].eq("").any():
        raise ValueError("Có lesson event thiếu content_id")
    if events.loc[events["event_name"].eq("join_campaign"), "campaign_id"].eq("").any():
        raise ValueError("Có join_campaign thiếu campaign_id")

    registrations = events["user_id"].map(users.set_index("user_id")["registered_at"])
    if (events["occurred_at"] < registrations).any():
        raise ValueError("Có sự kiện trước thời điểm đăng ký")
    if users["registered_at"].max().date() > covered_through or events["occurred_at"].max().date() > covered_through:
        raise ValueError("COVERED_THROUGH nhỏ hơn ngày đăng ký hoặc ngày sự kiện cuối")
    signups = events.loc[events["event_name"].eq("sign_up")]
    if (len(signups) != len(users) or signups["user_id"].duplicated().any()
            or not signups["occurred_at"].eq(registrations.loc[signups.index]).all()):
        raise ValueError("Mỗi user phải có đúng một sign_up tại registered_at")

    # Chuyển chuỗi trống thành None để pyodbc ghi SQL NULL, chỉ ở hai cột FK tùy chọn.
    for column in ("content_id", "campaign_id"):
        events[column] = events[column].astype(object).where(events[column].ne(""), None)
    return tables

clean = prepare_source(raw, COVERED_THROUGH)

quality = []
for table_name, frame in raw.items():
    for column in frame.columns:
        quality.append({
            "table": table_name,
            "column": column,
            "rows": len(frame),
            "blank_or_null_count": int(frame[column].fillna("").astype(str).str.strip().eq("").sum()),
            "duplicate_key_count": (int(frame[PRIMARY_KEYS[table_name]].duplicated().sum())
                                    if column == PRIMARY_KEYS[table_name] else ""),
        })
pd.DataFrame(quality).to_csv(PROCESSED_DIR / "data_quality_report.csv", index=False, encoding="utf-8-sig")
print("Data Quality PASS: 4 bảng, PK, event names, FK, timestamps, signup checks")
print(f"Events không có content_id: {int(raw['Events']['content_id'].eq('').sum()):,}")
print(f"Events không có campaign_id: {int(raw['Events']['campaign_id'].eq('').sum()):,}")
print("Đã xuất data/processed/data_quality_report.csv")

## 4. Hàm SQL Server ETL (được bảo vệ khỏi nạp đè ngoài ý muốn)

**Không có `DELETE` nào được chạy nếu `RUN_IMPORT=False`.** Khi import, toàn bộ thao tác chạy trong transaction; phát hiện lỗi sẽ `ROLLBACK`. Chỉ nạp đè nếu bật `REPLACE_EXISTING=True` và xác nhận đúng tên database. Tên SQL View và các cột KPI được giữ như dự án Power BI hiện tại.

In [ ]:
def sql_connection(database, *, autocommit=False):
    import pyodbc
    supported = ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server")
    driver = next((d for d in supported if d in pyodbc.drivers()), None)
    if driver is None:
        raise RuntimeError("Cài ODBC Driver 17/18 for SQL Server rồi chạy lại")
    conn_string = (
        f"DRIVER={{{driver}}};SERVER={{{SQL_SERVER}}};DATABASE={{{database}}};"
        "Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;"
    )
    return pyodbc.connect(conn_string, autocommit=autocommit, timeout=10)

def sql_batches(path):
    sql_text = path.read_text(encoding="utf-8-sig")
    # GO là dấu phân cách batch của SSMS, không phải câu lệnh T-SQL.
    return [part.strip() for part in re.split(r"(?im)^\s*GO\s*(?:--[^\n]*)?$", sql_text)
            if part.strip()]

def exec_sql_file(cursor, filename):
    for batch in sql_batches(PROJECT_DIR / "sql" / filename):
        cursor.execute(batch)

def python_rows(frame, columns):
    for row in frame.loc[:, columns].itertuples(index=False, name=None):
        yield tuple(v.to_pydatetime() if isinstance(v, pd.Timestamp) else v for v in row)

def load_database(tables):
    with closing(sql_connection("master", autocommit=True)) as master:
        if master.execute("SELECT DB_ID(?)", DATABASE).fetchval() is None:
            master.execute(f"CREATE DATABASE [{DATABASE}]")

    with closing(sql_connection(DATABASE, autocommit=False)) as db:
        cursor = db.cursor()
        try:
            exec_sql_file(cursor, "01_schema.sql")
            current = {name: cursor.execute(f"SELECT COUNT(*) FROM dbo.[{name}]").fetchval()
                       for name in (*SOURCE_NAMES, "DatasetCoverage")}
            if any(count > 0 for count in current.values()):
                if not REPLACE_EXISTING:
                    raise RuntimeError(
                        f"Database đã có dữ liệu {current}. Không nạp đè. "
                        "Giữ RUN_IMPORT=False hoặc chủ động bật REPLACE_EXISTING=True."
                    )
                if CONFIRM_REPLACEMENT != DATABASE:
                    raise RuntimeError("Bạn phải đặt CONFIRM_REPLACEMENT đúng bằng tên database để cho phép thay dữ liệu")

            # Thứ tự xóa bảo vệ các quan hệ FK; chỉ thực thi khi đã được xác nhận.
            for name in ("Events", "Users", "Content", "Campaigns", "DatasetCoverage"):
                cursor.execute(f"DELETE FROM dbo.[{name}]")

            loads = [
                ("Users", ("user_id", "registered_at", "acquisition_channel", "device_type", "learning_goal", "experience_level", "ho_ten", "tinh_thanh")),
                ("Content", ("content_id", "content_title", "topic", "level")),
                ("Campaigns", ("campaign_id", "campaign_name", "campaign_type", "start_date", "end_date")),
                ("Events", ("event_id", "user_id", "occurred_at", "event_name", "content_id", "campaign_id")),
            ]
            for name, cols in loads:
                col_list = ", ".join(f"[{x}]" for x in cols)
                placeholders = ", ".join("?" for _ in cols)
                cursor.executemany(
                    f"INSERT INTO dbo.[{name}] ({col_list}) VALUES ({placeholders})",
                    list(python_rows(tables[name], list(cols))),
                )
            cursor.execute("INSERT dbo.DatasetCoverage(dataset_id, covered_through) VALUES (1, ?)", COVERED_THROUGH)

            exec_sql_file(cursor, "02_user_metrics.sql")
            exec_sql_file(cursor, "03_content_campaign_views.sql")
            result = cursor.execute("""
                SELECT COUNT(*) AS users, SUM(activated_7d) AS activated,
                       SUM(retained_d7) AS retained, SUM(reached_complete) AS completed
                FROM dbo.vw_user_metrics
            """).fetchone()
            if result[0] != len(tables["Users"]):
                raise RuntimeError("Số dòng vw_user_metrics không khớp Users")
            db.commit()
            print(f"SQL ETL PASS: {len(tables['Users'])} Users, {len(tables['Events'])} Events, "
                  f"{len(tables['Content'])} Content, {len(tables['Campaigns'])} Campaigns")
            print(f"KPI: activated={result[1]}, retained D7={result[2]}, funnel complete={result[3]}")
        except Exception:
            db.rollback()
            raise

def refresh_views_only():
    with closing(sql_connection(DATABASE, autocommit=False)) as db:
        try:
            cursor = db.cursor()
            exec_sql_file(cursor, "01_schema.sql")
            exec_sql_file(cursor, "02_user_metrics.sql")
            exec_sql_file(cursor, "03_content_campaign_views.sql")
            db.commit()
            print("Đã tạo/cập nhật 3 view, không xóa hay nạp lại bảng dữ liệu")
        except Exception:
            db.rollback()
            raise

if RUN_IMPORT and RUN_VIEW_REFRESH:
    raise ValueError("Chỉ chọn một: RUN_IMPORT hoặc RUN_VIEW_REFRESH")
if RUN_IMPORT:
    load_database(clean)
elif RUN_VIEW_REFRESH:
    refresh_views_only()
else:
    print("Bỏ qua mọi thao tác ghi vào SQL Server (chế độ an toàn).")

## 5. Xuất dữ liệu user-level cho Power BI / đối chiếu khi cần

Bật `RUN_EXPORT=True` sau khi các View đã tồn tại trên SQL Server. File nguồn gốc không bị sửa. Kết quả tham khảo của dataset DUA Edu: **850 users / 609 activation / 202 D7 / 425 hoàn thành funnel**.

In [ ]:
if RUN_EXPORT:
    with closing(sql_connection(DATABASE)) as db:
        cursor = db.cursor()
        cursor.execute("SELECT * FROM dbo.vw_user_metrics ORDER BY user_id")
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
    if len(rows) != len(clean["Users"]):
        raise ValueError("Số dòng user metrics không khớp Users")
    output_file = PROCESSED_DIR / "user_metrics.csv"
    with output_file.open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.writer(stream)
        writer.writerow(columns)
        writer.writerows(rows)
    print("Xuất user-level metrics:", output_file, "—", len(rows), "users")
else:
    print("RUN_EXPORT=False: không truy vấn hoặc xuất SQL Server.")

## 6. Đối chiếu trong SSMS

Sau khi nạp hoặc cập nhật View, mở `sql/04_quality_checks.sql` và `sql/05_analysis_reports.sql` bằng SSMS. Đối chiếu Funnel `850 → 797 → 710 → 609 → 425`; Content `C001 205/166/91`; Campaign `CP02 63/21`. Các mốc này chỉ đúng với bản dữ liệu giả lập được cung cấp, không cố định trong công thức.

**Chú ý:** Nếu Power BI báo không thể kết nối, hãy kiểm tra tên server/cơ chế xác thực và thực hiện Refresh. Notebook không thay đổi nội dung file `.pbix`.